# exp4_inequity_aversion — Hughes et al. 2018 literature baseline

A single condition (`inequity_aversion`): DQN agents with the **same
observations as `D_res_full_empathy`** (`see_resource=True,
see_emotions=True`) but a completely different reward channel — the
subjective, inequity-averse reward of Hughes, Leibo et al. 2018
("Inequity aversion improves cooperation in intertemporal social dilemmas",
NeurIPS), instead of this project's own empathic-weight formula.

**Why this run exists.** R2 asked directly: *"How does your mechanism differ
from Hughes et al., and would you expect their equality result to survive
under your affective-signal channel?"* This notebook is the empirical answer
— the two mechanisms, same observations, same environment, compared on the
same two metrics.

**The two reward formulas, side by side** (`agent_policies_gt_ais.py`):

- **This project (`reward_mode="social"`)**, used by every other condition
  in this folder:
  `r_i = (1 − α_i)·personal_i + α_i·empathic_i`, where `empathic_i` is the
  mean of the *other* agents' emotion signal (a smoothed, bounded transform
  of their recent consumption rate — see the earlier discussion of
  `SocialRewardCalculator`).
- **Hughes et al. 2018 (`reward_mode="inequity_aversion"`)**, used here.
  Each agent keeps a temporally smoothed *raw* reward trace
  `e_i(t) = λ·e_i(t−1) + r_i(t)` (`λ = 0.95`) and receives
  `u_i(t) = r_i(t) − (α_ia/(N−1))·Σ_j max(e_j − e_i, 0) − (β_ia/(N−1))·Σ_j max(e_i − e_j, 0)`,
  i.e. the immediate meal outcome minus an **envy** penalty (weighted by
  `α_ia`, disadvantageous inequity) and a **guilt** penalty (weighted by
  `β_ia`, advantageous inequity). The default parameters used here —
  `α_ia = 0.0`, `β_ia = 0.5`, `λ = 0.95`
  (`PARAMS_IA` in `main_gt_ais.py`) — switch envy off entirely and only
  penalise an agent for eating *more* than its (smoothed) peers: guilt-only,
  as in Hughes et al.'s cooperation-promoting configuration.

**A reading note the CSV alone will not tell you.** The `inequity_aversion`
condition is created with `alpha=0.0` in `make_condition(...)`
(`main_gt_ais.py`), so the `alpha` column in its summary CSV reads `0.0` —
but this value is **never used**. `env_gt_ais.py` builds the
`SocialRewardCalculator` unconditionally (it still drives the observation's
emotion slot) but swaps in `InequityAversionCalculator` as
`self.reward_calculator` whenever `reward_mode == "inequity_aversion"`
(lines 117-122); the per-agent `alpha` is only ever read by
`SocialRewardCalculator.calculate_rewards`, which is not what computes the
reward here. Do not read the `alpha=0.0` column as "non-empathic" for this
condition — it is simply unused metadata.

**Two metrics, one comparison.** `inequity_aversion` vs. `D_res_full_empathy`
(loaded from `exp1_resource_obs`, the condition with matching observations
and the project's own α = 0.5 empathic reward) on:
- **Depletion step** (`total_steps`) — sustainability.
- **Gini coefficient of meals** (`gini_meals`) — fairness, the literature
  contrast R2 asked for.

As elsewhere, the final analysis uses the **greedy evaluation episodes
only** (`phase='eval'`, ε = 0, no learning). This notebook only reads
`csv_results/`; nothing is written to disk and no experiment code is
touched.

## 1. Setup

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

EXPERIMENT = "exp4_inequity_aversion"
RESULTS_DIR = Path("..") / "csv_results" / EXPERIMENT
COMPARISON_EXPERIMENT = "exp1_resource_obs"
COMPARISON_CONDITION = "D_res_full_empathy"
COMPARISON_DIR = Path("..") / "csv_results" / COMPARISON_EXPERIMENT

LAST_N_EVAL = 10
ALPHA_LEVEL = 0.05
TARGET_SEEDS = 20

print(f"experiment : {EXPERIMENT}  (data: {RESULTS_DIR.resolve()})")
print(f"comparison : {COMPARISON_CONDITION} from {COMPARISON_EXPERIMENT} "
      f"(data: {COMPARISON_DIR.resolve()})")

In [ ]:
# chart chrome: recessive grid and axes, text in neutral ink, series colour
# reserved for the marks themselves. Lighter than exp1/exp6's palette block
# since this notebook has at most one figure.
SURFACE = "#fcfcfb"
INK = {"primary": "#0b0b0b", "secondary": "#52514e", "muted": "#898781",
       "grid": "#e1e0d9", "baseline": "#c3c2b7"}
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "figure.dpi": 110, "font.size": 10,
    "axes.edgecolor": INK["baseline"], "axes.linewidth": 1.0,
    "axes.labelcolor": INK["secondary"], "axes.labelsize": 10,
    "axes.titlecolor": INK["primary"], "axes.titlesize": 11,
    "axes.titleweight": "semibold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "axes.axisbelow": True,
    "grid.color": INK["grid"], "grid.linewidth": 0.8,
    "xtick.color": INK["muted"], "ytick.color": INK["muted"],
    "xtick.labelcolor": INK["secondary"], "ytick.labelcolor": INK["secondary"],
    "text.color": INK["primary"], "legend.frameon": False, "legend.fontsize": 9,
})

METRIC_LABEL = {
    "depletion_step": "Depletion step (steps before the pool empties)",
    "gini_meals": "Gini coefficient of meals",
    "total_meals": "Total meals consumed per episode",
}

METRICS = ["depletion_step", "gini_meals"]

## 2. Load the runs

Two sources: this experiment's own folder, and — for the comparison — just
the `D_res_full_empathy` rows of `exp1_resource_obs`. If `exp1` has not been
run yet on this machine, the comparison cells below are skipped rather than
raising, so the descriptive sections of this notebook still work on their
own.

In [ ]:
def load_summaries(results_dir: Path) -> pd.DataFrame:
    """Concatenate every *_episode_summary.csv of one experiment folder.

    The runner writes all condition parameters as columns (main_gt_ais.py,
    summary_header()), so no filename parsing is needed.
    """
    files = sorted(results_dir.glob("*_episode_summary.csv"))
    if not files:
        raise FileNotFoundError(f"no *_episode_summary.csv under {results_dir.resolve()}")
    frames = []
    for path in files:
        one = pd.read_csv(path)
        one["source_file"] = path.name
        frames.append(one)
    return pd.concat(frames, ignore_index=True)


df = load_summaries(RESULTS_DIR)
print(f"{df['source_file'].nunique()} run files, {len(df)} episode rows\n")

inventory = (df.groupby("condition", observed=True)
               .agg(seeds=("seed", "nunique"), runs=("source_file", "nunique"),
                    eval_episodes=("phase", lambda s: int((s == "eval").sum())))
               .sort_index())
inventory["seeds_missing_vs_target"] = TARGET_SEEDS - inventory["seeds"]
display(inventory)

print(f"\nreward_mode in this file: {sorted(df['reward_mode'].unique())}")
print(f"alpha column values (unused metadata here, see the note above): "
      f"{sorted(df['alpha'].unique())}")

HAVE_COMPARISON = COMPARISON_DIR.exists() and any(COMPARISON_DIR.glob("*_episode_summary.csv"))
if HAVE_COMPARISON:
    df_cmp_full = load_summaries(COMPARISON_DIR)
    df_cmp = df_cmp_full[df_cmp_full["condition"] == COMPARISON_CONDITION].copy()
    if df_cmp.empty:
        print(f"\nNOTE: {COMPARISON_EXPERIMENT} has data but no "
              f"'{COMPARISON_CONDITION}' rows yet -- comparison sections will be skipped.")
        HAVE_COMPARISON = False
    else:
        print(f"\ncomparison source: {df_cmp['seed'].nunique()} seeds of "
              f"{COMPARISON_CONDITION}")
else:
    print(f"\nNOTE: {COMPARISON_EXPERIMENT} not found under {COMPARISON_DIR.resolve()} -- "
          f"comparison sections below will be skipped, only the descriptive results for "
          f"{EXPERIMENT} itself will run.")

## 3. Data integrity and derived metrics

Same checks as elsewhere: duplicated episodes, and right-censoring of the
depletion step. **Read the censoring rate here carefully** — under
`α_ia=0, β_ia=0.5` the guilt penalty can make eating unattractive enough
that agents rarely deplete the pool within `max_steps`, which would make
`depletion_step` close to uninformative (everything censored at the same
ceiling) and shift the interesting signal to `total_meals` instead.

In [ ]:
def add_derived(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    frame["depletion_step"] = frame["total_steps"]
    frame["censored"] = (frame["total_steps"] >= frame["max_steps"]) & (frame["resource_remaining"] > 0)
    meal_cols = [c for c in frame.columns if re.fullmatch(r"meals_\d+", c)]
    frame["total_meals"] = frame[meal_cols].sum(axis=1)
    return frame


df = add_derived(df)
ev = df[df["phase"] == "eval"].copy()

dupes = int(df.duplicated(subset=["condition", "seed", "phase", "episode"]).sum())
print(f"duplicate (condition, seed, phase, episode) rows : {dupes}")
assert dupes == 0, "duplicated episode rows -- check for a re-run written twice"

censoring = (ev.groupby("condition", observed=True)
               .agg(eval_episodes=("censored", "size"),
                    censored_rate=("censored", "mean"),
                    median_total_meals=("total_meals", "median"),
                    median_resource_remaining=("resource_remaining", "median"))
               .sort_index())
display(censoring.style.format({"censored_rate": "{:.1%}"}))

if censoring["censored_rate"].iloc[0] > 0.5:
    print(f"WARNING: most eval episodes never depleted the pool. depletion_step is largely "
          f"a ceiling effect here (most runs hit max_steps); total_meals and "
          f"resource_remaining below are the metrics that still discriminate.")

if HAVE_COMPARISON:
    df_cmp = add_derived(df_cmp)
    ev_cmp = df_cmp[df_cmp["phase"] == "eval"].copy()

## 4. End-of-training aggregation (seed-level)

One row per (condition, seed): mean over the last `LAST_N_EVAL` greedy
evaluation episodes, as in every other notebook here.

In [ ]:
def end_of_training(eval_df: pd.DataFrame, group_keys: list[str],
                    last_n: int = LAST_N_EVAL) -> pd.DataFrame:
    tail = (eval_df.sort_values("episode")
                   .groupby(group_keys + ["seed"], observed=True)
                   .tail(last_n))
    return (tail.groupby(group_keys + ["seed"], observed=True)
                .agg(depletion_step=("depletion_step", "mean"),
                     gini_meals=("gini_meals", "mean"),
                     total_meals=("total_meals", "mean"),
                     censored_frac=("censored", "mean"),
                     n_eval=("episode", "size"))
                .reset_index())


seed_ia = end_of_training(ev, ["condition"])
print(f"inequity_aversion: {len(seed_ia)} seed-level rows")
display(seed_ia[["seed", "depletion_step", "gini_meals", "total_meals",
                 "censored_frac"]].sort_values("seed").reset_index(drop=True))

if HAVE_COMPARISON:
    seed_d = end_of_training(ev_cmp, ["condition"])
    print(f"\n{COMPARISON_CONDITION}: {len(seed_d)} seed-level rows")
    display(seed_d[["seed", "depletion_step", "gini_meals", "total_meals"]]
            .sort_values("seed").reset_index(drop=True))

## 5. IA vs. D_res_full_empathy

The comparison R2 asked for: same observations, two different reward
mechanisms. Welch's t-test (unequal variances) and Mann-Whitney U are
reported together, with Hedges' g as the effect size — the same trio used
for the planned contrast in `exp1`'s §10.

In [ ]:
def hedges_g(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    na, nb = len(a), len(b)
    pooled = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    d = (a.mean() - b.mean()) / pooled
    g = d * (1 - 3 / (4 * (na + nb) - 9))
    se = np.sqrt((na + nb) / (na * nb) + g ** 2 / (2 * (na + nb - 2)))
    return float(g), float(g - 1.96 * se), float(g + 1.96 * se)


def stars(p):
    return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "ns"


if HAVE_COMPARISON:
    for metric in METRICS + ["total_meals"]:
        ia = seed_ia[metric].to_numpy()
        dd = seed_d[metric].to_numpy()
        welch = stats.ttest_ind(ia, dd, equal_var=False)
        u, p_u = stats.mannwhitneyu(ia, dd, alternative="two-sided")
        g, lo, hi = hedges_g(ia, dd)
        label = METRIC_LABEL.get(metric, metric)
        print("=" * 78)
        print(label)
        print(f"  inequity_aversion       : {ia.mean():8.3f}  (n={len(ia)})")
        print(f"  {COMPARISON_CONDITION:<24} : {dd.mean():8.3f}  (n={len(dd)})")
        print(f"  difference IA - D        : {ia.mean() - dd.mean():+8.3f}")
        print(f"  Welch t({welch.df:.1f}) = {welch.statistic:.3f}, p = {welch.pvalue:.4f} "
              f"{stars(welch.pvalue)}")
        print(f"  Mann-Whitney U = {u:.1f}, p = {p_u:.4f} {stars(p_u)}")
        print(f"  Hedges g = {g:+.2f}  [{lo:+.2f}, {hi:+.2f}]")
else:
    print("comparison skipped -- exp1_resource_obs / D_res_full_empathy not available")

## 6. Reading the comparison: two different routes to a "long-lived" pool

A high depletion step is not, by itself, evidence of the same kind of
sustainability in both conditions. `total_meals` and
`resource_remaining` separate two very different mechanisms that can both
produce a long episode:

- **A genuinely paced, shared consumption** (the mechanism this project's
  empathic reward is meant to produce): the pool is still consumed close to
  its full 500 units, just spread out over more steps.
- **Near-total abstention**: consumption collapses because the guilt penalty
  makes eating not worth it for most agents most of the time, so the pool
  is barely touched — a long or even uncensored episode here reflects
  inactivity, not paced sharing.

This is the same distinction `EXPERIMENT_TRACKER.md` flags for `exp6`'s
α = 0.99 condition (the personal reward term nearly extinguished) — worth
checking here too, since Hughes-style guilt aversion is a different
mechanism that could land on the same degenerate outcome for a different
reason.

In [ ]:
if HAVE_COMPARISON:
    consumption = pd.DataFrame({
        "condition": ["inequity_aversion", COMPARISON_CONDITION],
        "n_seeds": [len(seed_ia), len(seed_d)],
        "mean_total_meals": [seed_ia["total_meals"].mean(), seed_d["total_meals"].mean()],
        "median_total_meals": [seed_ia["total_meals"].median(), seed_d["total_meals"].median()],
        "share_near_zero_meals (<5% of pool)": [
            float((seed_ia["total_meals"] < 0.05 * 500).mean()),
            float((seed_d["total_meals"] < 0.05 * 500).mean()),
        ],
        "mean_depletion_step": [seed_ia["depletion_step"].mean(), seed_d["depletion_step"].mean()],
        "mean_gini": [seed_ia["gini_meals"].mean(), seed_d["gini_meals"].mean()],
    })
    display(consumption.style.format({c: "{:.3f}" for c in consumption.columns
                                      if consumption[c].dtype.kind == "f"}))
    if consumption.loc[0, "share_near_zero_meals (<5% of pool)"] > 0.3:
        print("NOTE: a substantial share of inequity_aversion seeds consume almost nothing "
              "-- read depletion_step and Gini for this condition in that light (a Gini "
              "computed on near-zero meal counts reflects a few outlier meals, not a shared "
              "equilibrium).")

## 7. Figure

In [ ]:
if HAVE_COMPARISON:
    order = ["inequity_aversion", COMPARISON_CONDITION]
    colors = {"inequity_aversion": CAT[3], COMPARISON_CONDITION: CAT[1]}
    frames = {"inequity_aversion": seed_ia, COMPARISON_CONDITION: seed_d}
    rng = np.random.default_rng(0)

    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2))
    for ax, metric in zip(axes, METRICS + ["total_meals"]):
        boxes = [frames[c][metric].to_numpy() for c in order]
        bp = ax.boxplot(boxes, positions=range(len(order)), widths=0.5, showfliers=False,
                        patch_artist=True)
        for patch, c in zip(bp["boxes"], order):
            patch.set_facecolor(colors[c]); patch.set_alpha(0.18); patch.set_edgecolor(colors[c])
        for med in bp["medians"]:
            med.set_color(INK["primary"]); med.set_linewidth(1.4)
        for i, c in enumerate(order):
            vals = frames[c][metric].to_numpy()
            ax.scatter(i + rng.uniform(-0.08, 0.08, len(vals)), vals, s=32,
                       color=colors[c], edgecolor=SURFACE, linewidth=1.0, zorder=3)
        ax.set_xticks(range(len(order))); ax.set_xticklabels(["IA", "D (social, alpha=0.5)"])
        ax.set_title(METRIC_LABEL.get(metric, metric).split(" (")[0])
    fig.suptitle("inequity_aversion vs. D_res_full_empathy -- seed-level means, last "
                 f"{LAST_N_EVAL} greedy eval episodes", x=0.012, ha="left", fontsize=11,
                 fontweight="semibold", color=INK["secondary"])
    fig.tight_layout()
    plt.show()
else:
    print("figure skipped -- comparison data not available")

## 8. Caveats

- **Seed count.** See §2's inventory against `TARGET_SEEDS`.
- **`β_ia` is not swept.** The tracker lists an optional β_ia sweep if this
  result is ambiguous — not run here; the comparison above uses only the
  advantageous-inequity-only default from Hughes et al.'s cooperation
  configuration (`PARAMS_IA` in `main_gt_ais.py`).
- **The `alpha=0.0` column for `inequity_aversion` is unused metadata** — see
  the reading note in §0. Do not interpret it as "non-empathic".
  